# PackTok: Semantic Packs in Google Colab

Dieses Notebook installiert den gepinnten Rust-/CUDA-Trainingspfad, bereitet Daten vor und führt einen begrenzten GPU-Preflight sowie einen Trainingslauf aus. Modell, Tokenizer, semantische Features und Checkpoints werden in Rust verarbeitet. Python steuert ausschließlich Installation und Colab.

**Start:** In Colab `Laufzeit → Laufzeittyp ändern → L4 GPU` wählen, unten Einstellungen prüfen und `Alle ausführen` starten. Die Standardeinstellung `tiny` mit Demo-Daten ist ein vollständiger Funktionscheck. `100m` und `1b` sind auswählbare größere Konfigurationen. Ein 1B-Lauf wurde hier nicht als auf L4 passend oder stabil nachgewiesen: der Preflight misst die reale Laufzeit und den Speicherbedarf und stoppt bei Problemen. Es erfolgt kein automatischer Wechsel auf eine andere GPU oder ein kleineres Modell.

Die Demo ist ein kleiner originaler synthetischer Deutsch-/Englisch-Korpus unter Apache-2.0. Er prüft die Pipeline, belegt keine Modellqualität oder Generalisierung. Bei `1b` und Demo-Daten läuft ausschließlich der Hardware-Preflight. Für echtes Training `USE_DEMO_DATA=False` setzen und einen Ordner mit getrennten `train.txt`, `validation.txt`, `test.txt` angeben. Es werden keine großen Datensätze automatisch heruntergeladen.

Checkpoints enthalten Modell, Optimizer und Fortschritt; beim 1B-Modell belegen sie viele GB. Rust schreibt sie auf die lokale Colab-Platte; ein externer Kopierer spiegelt ausschließlich vollständige Trainings-Generationen nach Google Drive und prüft SHA-256-Hashes. Derselbe `RUN_NAME` mit derselben Konfiguration ermöglicht Wiederaufnahme. Bei Fehlern bleiben die Logs erhalten. Nach bestätigter Drive-Sicherung wird die eigene Colab-Laufzeit standardmäßig getrennt. Bei fehlgeschlagener Sicherung bleibt sie zum Retten lokaler Ergebnisse verbunden. Ohne Drive wird nur mit `ALLOW_EPHEMERAL_DISCARD=True` automatisch getrennt. Setup und CUDA-Kompilierung können mehrere Minuten dauern. `1b` mit Drive erfordert `DRIVE_BUDGET_GB>=30` als ausdrücklich bereitgestellten Arbeitsraum: beim Ersetzen existieren alter und neuer Checkpoint gleichzeitig; ein kostenloses 15-GB-Drive reicht dafür nicht. Die Zahl ist eine Nutzerangabe, keine Abfrage der Drive-Quote. Ohne Drive sind 1B-Preflight und Training lokal möglich, ihre Ergebnisse sind dann flüchtig.


In [ ]:
# @title 1. Einstellungen
PRESET = "tiny" # @param ["tiny", "100m", "1b"]
VARIANT = "semantic" # @param ["baseline", "semantic"]
CONTEXT = 128 # @param {type:"integer"}
MICRO_BATCH = 1 # @param {type:"integer"}
ACCUMULATION = 4 # @param {type:"integer"}
STEPS = 100 # @param {type:"integer"}
TRAIN_MINUTES = 10 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
DTYPE = "bf16" # @param ["bf16", "f32"]
USE_DEMO_DATA = True # @param {type:"boolean"}
RAW_INPUT_DIR = "/content/drive/MyDrive/PackTok/data" # @param {type:"string"}
RUN_NAME = "semantic-tiny-42" # @param {type:"string"}
RESUME = True # @param {type:"boolean"}
MOUNT_DRIVE = True # @param {type:"boolean"}
DISCONNECT_AFTER_RUN = True # @param {type:"boolean"}
ALLOW_EPHEMERAL_DISCARD = False # @param {type:"boolean"}
DRIVE_BUDGET_GB = 10 # @param {type:"integer"}

# Gepinnter Implementierungsstand für Installation und Training.
SOURCE_COMMIT = "ead9f0a56b2ea382f64626a0a4f2d4c25626e837"


## 2. Installation und Ausführung

Google fragt beim ersten Start nach dem Drive-Zugriff. Danach läuft die Pipeline ohne manuelle Shell-Befehle. Zuerst wird der GPU-Typ geprüft, dann werden Rust 1.90 und bei Bedarf die offiziellen CUDA-12.8-Entwicklungswerkzeuge installiert. Bestehende Trainingsdaten oder Konfigurationen werden bei Abweichungen nicht überschrieben; wähle dafür einen neuen `RUN_NAME`.

Der erste GPU-Preflight hat ein 20-Minuten-Limit nach Installation/Kompilierung. Die Trainingsphase verwendet `TRAIN_MINUTES` (höchstens 20 in diesem Pilot-Notebook). Ein Zeitlimit ist kein Versprechen, dass eine volle große Checkpoint-Datei in dieser Zeit fertig wird. Eine unvollständige Generation ersetzt den letzten vollständigen Checkpoint nicht.


In [ ]:
# @title Installieren, prüfen, trainieren und Ergebnisse sichern
import importlib.util
import json
from pathlib import Path
import re
import shutil
import subprocess
import time
import traceback


def checkout_source(directory, commit, remote="https://github.com/DasEtwa/PackTok.git"):
    if re.fullmatch(r"[0-9a-f]{40}", commit) is None:
        raise RuntimeError("Ein unveränderlicher Release-Commit wird benötigt.")
    fresh = not directory.exists()
    if fresh:
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", remote, str(directory)], check=True)
    else:
        # A fresh --no-checkout clone has an empty index, so checking it against
        # HEAD before the first checkout would reject every fresh installation.
        subprocess.run(["git", "diff", "--quiet"], cwd=directory, check=True)
        subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=directory, check=True)
    subprocess.run(["git", "fetch", "--depth", "1", "origin", commit], cwd=directory, check=True)
    subprocess.run(["git", "checkout", "--detach", commit], cwd=directory, check=True)
    actual = subprocess.run(["git", "rev-parse", "HEAD"], cwd=directory,
                            check=True, capture_output=True, text=True).stdout.strip()
    if actual != commit:
        raise RuntimeError("Git-Quellstand stimmt nicht mit dem Notebook überein.")


def should_disconnect(state, mount_drive, allow_ephemeral_discard):
    return state.get("gpu_work_started", False) and (
        state.get("verified", False)
        or (not mount_drive and allow_ephemeral_discard)
    )


repo = Path("/content/PackTok-semantic-scale")
persistent = None
result = None
preservation = {"verified": False, "gpu_work_started": False}
try:
    if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
        raise RuntimeError("Das Notebook benötigt einen gepinnten Release-Commit; bitte die veröffentlichte Version laden.")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        persistent = Path("/content/drive/MyDrive/PackTok/semantic-scale")
        persistent.mkdir(parents=True, exist_ok=True)
    gpu = subprocess.run([
        "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
    ], check=True, capture_output=True, text=True).stdout.strip()
    if gpu not in ("NVIDIA L4", "NVIDIA L4 GPU"):
        raise RuntimeError(f"Gefunden: {gpu!r}. Bitte eine NVIDIA-L4-Laufzeit auswählen.")
    if shutil.which("git") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "git", "ca-certificates"], check=True)
    checkout_source(repo, SOURCE_COMMIT)
    module_path = repo / "experiments/semantic-scale/tools/colab_bootstrap.py"
    spec = importlib.util.spec_from_file_location("packtok_colab", module_path)
    bootstrap = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[spec.name] = bootstrap
    spec.loader.exec_module(bootstrap)
    settings = bootstrap.Settings(
        preset=PRESET, variant=VARIANT, context=CONTEXT,
        micro_batch=MICRO_BATCH, accumulation=ACCUMULATION,
        steps=STEPS, train_minutes=TRAIN_MINUTES, seed=SEED,
        dtype=DTYPE, demo=USE_DEMO_DATA, resume=RESUME, run_name=RUN_NAME,
    )
    result = bootstrap.execute_pipeline(
        repo, persistent, settings, Path(RAW_INPUT_DIR),
        drive_budget_gb=DRIVE_BUDGET_GB, preservation=preservation,
    )
    print("\nErgebnisse: " + result["run_directory"])
    print("Rust schreibt lokal; nur vollständige geprüfte Trainings-Checkpoints werden nach Drive gespiegelt.")
except BaseException as error:
    print("PackTok-Lauf gestoppt: " + str(error))
    failure = Path("/content/packtok-notebook-failures") / (str(time.time_ns()) + ".json")
    failure.parent.mkdir(parents=True, exist_ok=True)
    failure.write_text(json.dumps({"error": str(error), "traceback": traceback.format_exc(),
                                   "source_commit": SOURCE_COMMIT, "run_name": RUN_NAME},
                                  indent=2, ensure_ascii=False) + "\n")
    print("Lokaler Fehlerbericht: " + str(failure))
    if persistent is not None:
        try:
            destination = persistent / "notebook-failures" / failure.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(failure, destination)
        except OSError as backup_error:
            preservation["verified"] = False
            print("Drive-Fehlerbericht konnte nicht kopiert werden: " + str(backup_error))
    raise
finally:
    safe_to_disconnect = should_disconnect(preservation, MOUNT_DRIVE, ALLOW_EPHEMERAL_DISCARD)
    if DISCONNECT_AFTER_RUN and safe_to_disconnect:
        print("Ergebnisse sind gesichert oder flüchtiges Verwerfen wurde ausdrücklich gewählt. Die eigene Colab-Laufzeit wird getrennt.", flush=True)
        from google.colab import runtime
        runtime.unassign()
    elif DISCONNECT_AFTER_RUN:
        print("Laufzeit bleibt verbunden: keine bestätigte Sicherung. Lokale Ergebnisse zuerst sichern und anschließend in Colab trennen.", flush=True)


## Ergebnisse und Wiederaufnahme

Lokal unter `/content/packtok-runs/<RUN_NAME>/` liegen Konfiguration, GPU-Preflight, Trainingsmetriken, Auswertung und Checkpoints. Bei aktiviertem Drive liegen geprüfte Kopien unter `MyDrive/PackTok/semantic-scale/runs/<RUN_NAME>/`. Rust behält lokal zwei vollständige Trainings-Generationen und eine Preflight-Generation; Drive behält nach bestätigter Kopie die neueste Trainings-Generation. Protokolle bleiben erhalten. Diagnostische Preflight-Modellbilder und vorbereitete Korpora werden nicht gespiegelt. Unter `notebook-attempts/` bleiben Installations-/Build-Logs und Anfragen der einzelnen Starts erhalten. Notebook-Fehler liegen lokal in `/content/packtok-notebook-failures/` und werden bei funktionierendem Drive zusätzlich nach `notebook-failures/` kopiert.

Für Wiederaufnahme dieselbe Notebook-Version, denselben `RUN_NAME`, dieselben Daten und Einstellungen verwenden und `RESUME=True` lassen. Mit neuen Daten, neuer Modellgröße oder einer anderen Variante einen neuen Namen wählen. Ein 1B-Demo-Preflight erzeugt keinen Nachweis sinnvoller Vortrainingsqualität; für lange Läufe müssen Datenumfang, Hardware und Budget gesondert geplant werden.

Für einen Vergleich `baseline` und `semantic` mit denselben Daten, Modellgrößen, Seeds und Schrittbudgets in getrennten Laufverzeichnissen ausführen. Ein einzelner Durchlauf beweist keinen Vorteil der Semantic Packs. Die semantischen Features verwenden ausschließlich bereits vollständig beobachtete Ausdrücke; Kontextauflösungen, die zukünftige Wörter benötigen, werden im v1-Modelladapter ausgelassen.
